# OmniSVG — Text to SVG in Google Colab

> **Before running:** Runtime → Change runtime type → **T4 GPU**

| Model | Size | VRAM |
|-------|------|------|
| OmniSVG1.1_4B (4-bit on T4) | ~4 GB | Free tier ✅ |
| OmniSVG1.1_4B (bf16) | ~7.7 GB | A100 / L4 ✅ |
| OmniSVG1.1_8B (bf16) | ~17 GB | A100 ✅ |

Paper: https://arxiv.org/abs/2504.06263

In [ ]:
# Cell 1 — Verify GPU
import subprocess, sys
r = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True
)
if r.returncode == 0:
    print('GPU:', r.stdout.strip())
else:
    raise RuntimeError('No GPU. Runtime → Change runtime type → T4 GPU')


In [ ]:
# Cell 2 — Install system + Python dependencies
import os

# Cairo (required by cairosvg on Linux/Colab)
os.system('apt-get install -y -q libcairo2 libcairo2-dev 2>/dev/null')

os.system(
    'pip install -q '
    'transformers==4.51.3 '
    'qwen-vl-utils==0.0.11 '
    'cairosvg==2.7.1 '
    'einops==0.4.1 '
    'moviepy==1.0.3 '
    'shapely==2.0.7 '
    'accelerate '
    'bitsandbytes '
    'huggingface_hub '
    'pyyaml '
    'Pillow '
)
print('All dependencies installed.')


In [ ]:
# Cell 3 — Clone OmniSVG repo
import os, sys

REPO_DIR = '/content/OmniSVG'
if not os.path.exists(REPO_DIR):
    os.system(f'git clone --depth=1 https://github.com/OmniSVG/OmniSVG.git {REPO_DIR}')
    print('Cloned.')
else:
    print('Already cloned.')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('cwd:', os.getcwd())


In [ ]:
# Cell 4 — Download model weights
# First run: ~7.7 GB download. Subsequent runs: served from cache.
from huggingface_hub import hf_hub_download

MODEL_SIZE   = '4B'                        # change to '8B' for the larger model
QWEN_REPO    = 'Qwen/Qwen2.5-VL-3B-Instruct'
OMNISVG_REPO = 'OmniSVG/OmniSVG1.1_4B'

# Uncomment for 8B model (needs A100):
# MODEL_SIZE   = '8B'
# QWEN_REPO    = 'Qwen/Qwen2.5-VL-7B-Instruct'
# OMNISVG_REPO = 'OmniSVG/OmniSVG1.1_8B'

print(f'Fetching {OMNISVG_REPO} weights (~7.7 GB) ...')
bin_path = hf_hub_download(
    repo_id=OMNISVG_REPO,
    filename='pytorch_model.bin',
    resume_download=True,
)
print('Weights at:', bin_path)


In [ ]:
# Cell 5 — Load the model (runs once, ~1-2 min)
import os, gc, io, yaml, torch, numpy as np
from PIL import Image
from pathlib import Path
from transformers import AutoProcessor
from decoder import SketchDecoder
from tokenizer import SVGTokenizer

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
CONFIG_PATH = os.path.join(REPO_DIR, 'config.yaml')

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

dtype    = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
use_4bit = vram_gb < 20
print(f'GPU: {torch.cuda.get_device_name(0)}  ({vram_gb:.1f} GB)  4-bit={use_4bit}')

print('Loading processor ...')
processor = AutoProcessor.from_pretrained(
    QWEN_REPO, padding_side='left', trust_remote_code=True
)
processor.tokenizer.padding_side = 'left'

print('Loading Qwen backbone ...')
model = SketchDecoder(
    config_path=CONFIG_PATH, model_path=QWEN_REPO,
    model_size=MODEL_SIZE, torch_dtype=dtype, use_4bit=use_4bit,
)

print('Applying OmniSVG adapter weights ...')
state = torch.load(bin_path, map_location='cpu', weights_only=False)
missing, _ = model.load_state_dict(state, strict=False)
print(f'  missing keys: {len(missing)} (normal for open-source release)')
model.eval()

svg_tok = SVGTokenizer(CONFIG_PATH, model_size=MODEL_SIZE)

mc = cfg['model']
BOS, EOS, PAD = mc['bos_token_id'], mc['eos_token_id'], mc['pad_token_id']
BLACK_TOKEN = cfg['colors'].get('black_color_token', cfg['colors']['color_token_start'] + 2)
TARGET_SIZE = cfg['image']['target_size']
RENDER_SIZE = cfg['image'].get('render_size', 1024)

print('\n✅ Model ready!')


In [ ]:
# Cell 6 — generate_svg() function
import cairosvg
from IPython.display import display, SVG as IPySVG

SYSTEM_PROMPT = (
    'You are an expert SVG code generator. '
    'Generate precise, valid SVG path commands that accurately represent '
    'the described scene or object. '
    'Focus on capturing key shapes, spatial relationships, and visual composition.'
)

ICON_KW = {
    'icon', 'logo', 'symbol', 'badge', 'button', 'emoji', 'glyph', 'simple',
    'arrow', 'triangle', 'circle', 'square', 'heart', 'star', 'checkmark',
}
ILLUS_KW = {
    'illustration', 'scene', 'person', 'people', 'character', 'man', 'woman',
    'boy', 'girl', 'avatar', 'portrait', 'face', 'head', 'body', 'cat', 'dog',
    'bird', 'animal', 'pet', 'fox', 'rabbit', 'sitting', 'standing', 'walking',
    'running', 'sleeping', 'holding', 'playing', 'house', 'building', 'tree',
    'garden', 'landscape', 'mountain', 'forest', 'city', 'ocean', 'beach', 'sunset',
}
DEFAULTS = {
    'icon':         dict(temperature=0.5, top_p=0.88, top_k=50, repetition_penalty=1.05),
    'illustration': dict(temperature=0.6, top_p=0.90, top_k=60, repetition_penalty=1.03),
}


def _subtype(text):
    lower = text.lower()
    if any(k in lower for k in ICON_KW):
        return 'icon'
    if sum(1 for k in ILLUS_KW if k in lower) >= 1 or len(text) > 50:
        return 'illustration'
    return 'icon'


def _embed_dev():
    m = model.transformer
    if hasattr(m, 'model') and hasattr(m.model, 'embed_tokens'):
        return next(m.model.embed_tokens.parameters()).device
    return next(m.parameters()).device


def _render(svg_str, size=RENDER_SIZE):
    try:
        png = cairosvg.svg2png(bytestring=svg_str.encode(), output_width=size, output_height=size)
        rgba = Image.open(io.BytesIO(png)).convert('RGBA')
        bg = Image.new('RGB', rgba.size, (255, 255, 255))
        bg.paste(rgba, mask=rgba.split()[3])
        return bg
    except Exception:
        return None


def _valid(svg_str, img, sub):
    if not svg_str or len(svg_str) < 20 or '<svg' not in svg_str or img is None:
        return False
    k = 'empty_threshold_illustration' if sub == 'illustration' else 'empty_threshold_icon'
    return np.array(img).mean() < cfg['image'][k]


def generate_svg(
    prompt,
    max_new_tokens=1536,
    num_candidates=4,
    temperature=None,
    top_p=None,
    top_k=None,
    repetition_penalty=None,
    display_result=True,
    save_path=None,
):
    """
    Generate an SVG from a text prompt.

    Parameters
    ----------
    prompt          : str   Natural-language description of the SVG.
    max_new_tokens  : int   Max tokens per candidate (default 1536).
    num_candidates  : int   Max attempts; returns first valid one (default 4).
    display_result  : bool  Show SVG inline in Colab (default True).
    save_path       : str   Save SVG to this path if provided.

    Returns
    -------
    str | None  :  SVG string if successful, else None.
    """
    sub  = _subtype(prompt)
    defs = DEFAULTS[sub]
    temperature        = temperature        or defs['temperature']
    top_p              = top_p              or defs['top_p']
    top_k              = top_k              or defs['top_k']
    repetition_penalty = repetition_penalty or defs['repetition_penalty']

    print(f'Prompt  : {prompt}')
    print(f'Subtype : {sub}  |  temp={temperature}  top_p={top_p}  top_k={top_k}')

    instruction = (
        f'Generate an SVG illustration for: {prompt}\n\n'
        'Requirements:\n'
        '- Create complete SVG path commands\n'
        '- Include proper coordinates and colors\n'
        '- Maintain visual clarity and composition'
    )
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': [{'type': 'text', 'text': instruction}]},
    ]
    text_in = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs  = processor(text=[text_in], padding=True, truncation=True, return_tensors='pt')

    dev       = _embed_dev()
    input_ids = inputs['input_ids'].to(dev)
    attn_mask = inputs['attention_mask'].to(dev)

    print(f'Generating {num_candidates} candidate(s) x {max_new_tokens} tokens ...')
    with torch.no_grad():
        out = model.transformer.generate(
            input_ids=input_ids,
            attention_mask=attn_mask,
            max_new_tokens=max_new_tokens,
            num_return_sequences=num_candidates,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            top_k=int(top_k),
            repetition_penalty=repetition_penalty,
            early_stopping=True,
            eos_token_id=EOS,
            pad_token_id=PAD,
            bos_token_id=BOS,
            use_cache=True,
        )
    new_ids = out[:, input_ids.shape[1]:]

    for i in range(new_ids.shape[0]):
        try:
            ids = new_ids[i:i+1].cpu()
            wrapped = torch.cat(
                [torch.full((1,1), BOS), ids, torch.full((1,1), EOS)], dim=1
            )
            xy = svg_tok.process_generated_tokens(wrapped)
            if not len(xy):
                continue
            svg_t, col_t = svg_tok.raster_svg(xy)
            if not svg_t or not svg_t[0]:
                continue
            n = len(svg_t[0])
            while len(col_t) < n:
                col_t.append(BLACK_TOKEN)
            svg_obj = svg_tok.apply_colors_to_svg(svg_t[0], col_t)
            svg_str = svg_obj.to_str()
            if 'width=' not in svg_str:
                svg_str = svg_str.replace('<svg', f'<svg width="{TARGET_SIZE}" height="{TARGET_SIZE}"', 1)
            img = _render(svg_str)
            if not _valid(svg_str, img, sub):
                print(f'  [{i+1}] invalid (blank image)')
                continue
            print(f'  [{i+1}] valid — {n} paths')
            if save_path:
                Path(save_path).write_text(svg_str, encoding='utf-8')
                print(f'  Saved → {save_path}')
            if display_result:
                display(IPySVG(data=svg_str))
                if img:
                    display(img.resize((400, 400)))
            gc.collect()
            torch.cuda.empty_cache()
            return svg_str
        except Exception as e:
            print(f'  [{i+1}] error: {e}')

    print('No valid SVG. Try a different prompt or raise num_candidates.')
    gc.collect()
    torch.cuda.empty_cache()
    return None


print('✅ generate_svg() is ready. Run the next cell to generate!')


In [ ]:
# Cell 7 — Run example prompts
examples = [
    'A red heart icon with smooth curved edges, centered',
    'Simple house: red triangular roof, beige wall, brown door, two blue windows',
    'Cute orange cat sitting, round head with triangular ears, black outlines',
    'A green star badge with five points',
    'Coffee mug with three wavy steam lines, simple flat style',
]

for prompt in examples:
    print('=' * 64)
    safe = prompt[:30].replace(' ', '_')
    generate_svg(
        prompt,
        max_new_tokens=1536,
        num_candidates=4,
        display_result=True,
        save_path=f'/content/{safe}.svg',
    )
    print()


In [ ]:
# Cell 8 — Your custom prompt (edit and re-run)
MY_PROMPT = 'A minimalist fox logo facing right, triangular orange head, pointed ears'  # <- edit

svg = generate_svg(
    MY_PROMPT,
    max_new_tokens=1536,
    num_candidates=4,
    display_result=True,
    save_path='/content/my_output.svg',
)


In [ ]:
# Cell 9 — Download all generated SVGs
import glob
from google.colab import files

svgs = glob.glob('/content/*.svg')
print(f'Found {len(svgs)} SVG file(s):')
for f in svgs:
    print(' ', f)
    files.download(f)
